In [ ]:
import re
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.20.0


In [ ]:
df = pd.read_csv("/content/IMDB Dataset.csv")
print("Shape:", df.shape)
df.head(3)


Shape: (50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive


In [ ]:
df.duplicated().sum()

np.int64(418)

In [ ]:
df.drop_duplicates(inplace  = True)

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)        # remove HTML tags e.g. <br />
    text = re.sub(r"[^a-z\s]", " ", text)     # keep only letters
    text = re.sub(r"\s+", " ", text).strip()  # collapse whitespace
    return text

In [ ]:
df["clean_review"] = df["review"].apply(clean_text)

In [ ]:
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(df["clean_review"], df["label"],test_size=0.2, random_state=42, stratify=df["label"])

## 6. Prepare Sequences for the Neural Networks

### Tokenization
Convert each cleaned review into a sequence of integers, where each integer represents a word's rank in the vocabulary (fit **only** on the training set).

### Padding
Neural networks need fixed-length input. We pad/truncate every sequence to `max_len=200`.


In [ ]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=200, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=200, padding="post", truncating="post")

y_train_arr = y_train.values
y_test_arr = y_test.values


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

lstm_model = Sequential([
    Embedding(input_dim=10000, output_dim=64, mask_zero=True),

    LSTM(64),

    Dropout(0.3),

    Dense(32, activation="relu"),

    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history_lstm = lstm_model.fit(
    X_train_pad,
    y_train_arr,
    validation_split=0.1,
    epochs=15,
    batch_size=256,
    callbacks=[early_stop]
)

Epoch 1/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 99s 648ms/step - accuracy: 0.6910 - loss: 0.5576 - val_accuracy: 0.8596 - val_loss: 0.3659
Epoch 2/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 138s 622ms/step - accuracy: 0.8782 - loss: 0.3003 - val_accuracy: 0.8780 - val_loss: 0.3286
Epoch 3/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 88s 629ms/step - accuracy: 0.9037 - loss: 0.2514 - val_accuracy: 0.8717 - val_loss: 0.3033
Epoch 4/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 86s 614ms/step - accuracy: 0.9037 - loss: 0.2489 - val_accuracy: 0.8568 - val_loss: 0.3309
Epoch 5/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 88s 628ms/step - accuracy: 0.9142 - loss: 0.2257 - val_accuracy: 0.8624 - val_loss: 0.3335
Epoch 6/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 87s 623ms/step - accuracy: 0.9243 - loss: 0.2060 - val_accuracy: 0.8568 - val_loss: 0.3418
Epoch 7/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 144s 639ms/step - accuracy: 0.9264 - loss: 0.2010 - val_accuracy: 0.8694 - val_loss: 0.3613


In [ ]:
y_pred = (lstm_model.predict(X_test_pad) > 0.5).astype(int)

print(classification_report(y_test_arr, y_pred))

310/310 ━━━━━━━━━━━━━━━━━━━━ 11s 36ms/step
              precision    recall  f1-score   support

           0       0.89      0.84      0.86      4940
           1       0.85      0.90      0.87      4977

    accuracy                           0.87      9917
   macro avg       0.87      0.87      0.87      9917
weighted avg       0.87      0.87      0.87      9917



In [ ]:
accuracy_score(y_test_arr,y_pred)

0.8689119693455682